# ScamSense — Notebook 11: Drift Detection, MLflow Re-logging & CI/CD

**Platform:** Kaggle (CPU is fine — no GPU needed)

| Cell | Description |
|---|---|
| 1 | Install dependencies |
| 2 | Load `model_comparison.csv` |
| 3 | Evidently drift report (mBERT → XLM-RoBERTa) |
| 4 | Re-log metrics to MLflow on DagsHub |
| 5 | GitHub Actions CI/CD workflow (written to file) |
| 6 | GCP Cloud Run — architecture notes + Dockerfile |
| 7 | Save outputs |

**Kaggle Secrets required:** `DAGSHUB_TOKEN`

**Input:** `model_comparison.csv` (attach as Kaggle Dataset)

**Output:** `drift_report.html`, `github_actions_workflow.yml`, `Dockerfile` in `/kaggle/working/`

> **GCP Cloud Run** is documented here but not executed — no GCP account needed. It will be wired up in NB12 once the Telegram bot (NB09) is complete.

## Cell 1 — Install dependencies

In [10]:
# evidently>=0.5 uses a new API — pin to ensure compatibility
!pip install -q "evidently>=0.7" mlflow dagshub

import os
import pandas as pd
import numpy as np
from pathlib import Path

INPUT_DIR  = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working')

# Verify evidently version
import evidently
print('evidently version:', evidently.__version__)
print('Dependencies installed ✅')


evidently version: 0.7.21
Dependencies installed ✅


## Cell 2 — Load model_comparison.csv

In [11]:
# Auto-detect model_comparison.csv across all attached Kaggle datasets
def find_input_file(filename):
    matches = list(INPUT_DIR.rglob(filename))
    return matches[0] if matches else None

csv_path = find_input_file('model_comparison.csv')
if csv_path is None:
    raise FileNotFoundError(
        'model_comparison.csv not found in /kaggle/input.\n'
        'Upload it as a Kaggle Dataset and attach under Data → Add data.'
    )

df = pd.read_csv(csv_path)
print('Loaded:', csv_path)
print(f'Shape: {df.shape}')
print(df)

# Split into per-model frames
MODELS = df['model'].unique().tolist()
print('\nModels found:', MODELS)

# Reference = mBERT (baseline), Current = XLM-RoBERTa (best)
REF_MODEL = 'mbert-scamsense-baseline'
CUR_MODEL = 'xlmroberta-scamsense'

df_ref = df[df['model'] == REF_MODEL].copy().reset_index(drop=True)
df_cur = df[df['model'] == CUR_MODEL].copy().reset_index(drop=True)

print(f'\nReference ({REF_MODEL}): {len(df_ref)} rows')
print(f'Current   ({CUR_MODEL}): {len(df_cur)} rows')


Loaded: /kaggle/input/datasets/bhoovika/scamscene-model/model_comparison.csv
Shape: (12, 7)
                       model  language  accuracy      f1  precision  recall  \
0       xlmroberta-scamsense   overall    0.9916  0.9916     0.9916  0.9916   
1       xlmroberta-scamsense        en    0.9892  0.9892     0.9892  0.9892   
2       xlmroberta-scamsense        ms    1.0000  1.0000     1.0000  1.0000   
3       xlmroberta-scamsense  singlish    0.9991  0.9991     0.9991  0.9991   
4       xlmroberta-scamsense        ta    0.9993  0.9993     0.9993  0.9993   
5       xlmroberta-scamsense        zh    0.9991  0.9991     0.9991  0.9991   
6   mbert-scamsense-baseline   overall    0.9369  0.9369     0.9369  0.9369   
7   mbert-scamsense-baseline        en    0.9190  0.9190     0.9190  0.9190   
8   mbert-scamsense-baseline        ms    1.0000  1.0000     1.0000  1.0000   
9   mbert-scamsense-baseline  singlish    0.9856  0.9856     0.9858  0.9856   
10  mbert-scamsense-baseline        ta 

## Cell 3 — Evidently Drift Report

Evidently compares **mBERT (reference/baseline)** vs **XLM-RoBERTa (current/production)** across the metric columns (f1, precision, recall, auc, accuracy) per language slice.

This simulates how you would monitor for **model performance drift** in production — e.g. if a new model version underperforms the baseline on any language.

In [12]:
# evidently v0.7+ API — uses Dataset + DataDefinition instead of Report(metrics=[])
from evidently.presets import DataDriftPreset
from evidently.metrics import ValueDrift, DriftedColumnsCount
from evidently import Dataset, DataDefinition, Report

METRIC_COLS = ['accuracy', 'f1', 'precision', 'recall', 'auc']

# Drop 'overall' row — keep only per-language slices for drift computation
ref_data = df_ref[df_ref['language'] != 'overall'][METRIC_COLS].reset_index(drop=True)
cur_data = df_cur[df_cur['language'] != 'overall'][METRIC_COLS].reset_index(drop=True)

print('Reference (mBERT) per-language rows:')
print(ref_data)
print('\nCurrent (XLM-RoBERTa) per-language rows:')
print(cur_data)

# Wrap in Evidently Dataset
definition = DataDefinition(numerical_columns=METRIC_COLS)
ref_ds = Dataset.from_pandas(ref_data, data_definition=definition)
cur_ds = Dataset.from_pandas(cur_data, data_definition=definition)

# Build and run report
report = Report([
    DriftedColumnsCount(),
] + [ValueDrift(column=col) for col in METRIC_COLS])

result = report.run(reference_data=ref_ds, current_data=cur_ds)

# Save HTML report
report_path = OUTPUT_DIR / 'drift_report.html'
result.save_html(str(report_path))
print(f'\nDrift report saved ✅ -> {report_path}')

# Parse results
metrics_list = result.dict()['metrics']

# DriftedColumnsCount is first metric
drifted_val   = metrics_list[0]['value']
n_drifted     = int(drifted_val['count'])
drift_share   = float(drifted_val['share'])
drift_detected = n_drifted > 0

print('\n' + '═'*55)
print(f'Drifted columns  : {n_drifted} / {len(METRIC_COLS)}')
print(f'Drift share      : {drift_share:.0%}')
print(f'Drift detected   : {drift_detected}')
print()
print('Interpretation:')
if drift_detected:
    print('  ⚠️  XLM-RoBERTa metrics differ significantly from mBERT baseline.')
    print('  This is EXPECTED — XLM-RoBERTa genuinely outperforms mBERT.')
    print('  In production, unexpected drift would trigger a retraining alert.')
else:
    print('  ✅ No significant drift — models perform similarly across all metrics.')
    print('  K-S test p-values are above 0.05 threshold for all columns.')
print('═'*55)

# Per-metric summary (metrics 1 onwards are ValueDrift results)
print(f'\nPer-metric drift (K-S p-value, threshold=0.05):')
print(f'  {"-"*45}')
print(f'  {"Metric":<12} {"p-value":>10}  {"Drifted?"}')
print(f'  {"-"*45}')
for i, col in enumerate(METRIC_COLS):
    p_val   = float(metrics_list[i + 1]['value'])
    drifted = p_val < 0.05
    flag    = '⚠️  YES' if drifted else '✅  no'
    print(f'  {col:<12} {p_val:>10.4f}  {flag}')
print(f'  {"-"*45}')
print('  (p < 0.05 = statistically significant drift)')


Reference (mBERT) per-language rows:
   accuracy      f1  precision  recall     auc
0    0.9190  0.9190     0.9190  0.9190  0.9727
1    1.0000  1.0000     1.0000  1.0000  1.0000
2    0.9856  0.9856     0.9858  0.9856  0.9997
3    0.9993  0.9993     0.9993  0.9993  1.0000
4    0.9913  0.9913     0.9914  0.9913  0.9997

Current (XLM-RoBERTa) per-language rows:
   accuracy      f1  precision  recall     auc
0    0.9892  0.9892     0.9892  0.9892  0.9986
1    1.0000  1.0000     1.0000  1.0000  1.0000
2    0.9991  0.9991     0.9991  0.9991  0.9991
3    0.9993  0.9993     0.9993  0.9993  1.0000
4    0.9991  0.9991     0.9991  0.9991  1.0000


/usr/local/lib/python3.12/dist-packages/scipy/stats/_stats_py.py:7400: RuntimeWarning:

divide by zero encountered in divide

/usr/local/lib/python3.12/dist-packages/scipy/stats/_stats_py.py:7400: RuntimeWarning:

divide by zero encountered in divide




Drift report saved ✅ -> /kaggle/working/drift_report.html

═══════════════════════════════════════════════════════
Drifted columns  : 1 / 5
Drift share      : 20%
Drift detected   : True

Interpretation:
  ⚠️  XLM-RoBERTa metrics differ significantly from mBERT baseline.
  This is EXPECTED — XLM-RoBERTa genuinely outperforms mBERT.
  In production, unexpected drift would trigger a retraining alert.
═══════════════════════════════════════════════════════

Per-metric drift (K-S p-value, threshold=0.05):
  ---------------------------------------------
  Metric          p-value  Drifted?
  ---------------------------------------------
  accuracy         0.8730  ✅  no
  f1               0.8730  ✅  no
  precision        0.8730  ✅  no
  recall           0.8730  ✅  no
  auc              0.0000  ⚠️  YES
  ---------------------------------------------
  (p < 0.05 = statistically significant drift)


## Cell 4 — Re-log metrics to MLflow on DagsHub

Logs all rows from `model_comparison.csv` to your existing DagsHub MLflow experiment `scamsense-classification`. Each language slice gets its own nested run so you can filter by language in the DagsHub UI.

In [13]:
import mlflow
import dagshub
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
DAGSHUB_TOKEN    = secrets.get_secret('DAGSHUB_TOKEN')
DAGSHUB_USERNAME = 'Bhoovika'

os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN

dagshub.init(
    repo_owner=DAGSHUB_USERNAME,
    repo_name='ScamSense',
    mlflow=True,
)
mlflow.set_tracking_uri('https://dagshub.com/Bhoovika/ScamSense.mlflow')
mlflow.set_experiment('scamsense-classification')

print('MLflow connected ✅')
print('Experiment: scamsense-classification')
print('View at: https://dagshub.com/Bhoovika/ScamSense.mlflow\n')

# Log each model as a parent run, each language slice as a nested run
for model_name in [REF_MODEL, CUR_MODEL]:
    model_df = df[df['model'] == model_name]
    overall  = model_df[model_df['language'] == 'overall'].iloc[0]

    with mlflow.start_run(run_name=f'{model_name}-summary') as parent_run:
        # Log overall metrics on the parent run
        mlflow.set_tag('model', model_name)
        mlflow.set_tag('source', 'model_comparison.csv')
        for col in METRIC_COLS:
            mlflow.log_metric(f'overall_{col}', float(overall[col]))

        # Log per-language metrics as nested runs
        lang_df = model_df[model_df['language'] != 'overall']
        for _, row in lang_df.iterrows():
            lang = row['language']
            with mlflow.start_run(run_name=f'{model_name}-{lang}', nested=True):
                mlflow.set_tag('model', model_name)
                mlflow.set_tag('language', lang)
                for col in METRIC_COLS:
                    mlflow.log_metric(col, float(row[col]))

        print(f'Logged {model_name}: overall + {len(lang_df)} language runs')

print('\n✅ All metrics logged to DagsHub MLflow')
print('View: https://dagshub.com/Bhoovika/ScamSense.mlflow')


Initialized MLflow to track repo "Bhoovika/ScamSense"

Repository Bhoovika/ScamSense initialized!

MLflow connected ✅
Experiment: scamsense-classification
View at: https://dagshub.com/Bhoovika/ScamSense.mlflow

🏃 View run mbert-scamsense-baseline-en at: https://dagshub.com/Bhoovika/ScamSense.mlflow/#/experiments/0/runs/e50f147cc01d4edc84a355746c2e7b2c
🧪 View experiment at: https://dagshub.com/Bhoovika/ScamSense.mlflow/#/experiments/0
🏃 View run mbert-scamsense-baseline-ms at: https://dagshub.com/Bhoovika/ScamSense.mlflow/#/experiments/0/runs/32d13f52390440ee8151e5aae72355ec
🧪 View experiment at: https://dagshub.com/Bhoovika/ScamSense.mlflow/#/experiments/0
🏃 View run mbert-scamsense-baseline-singlish at: https://dagshub.com/Bhoovika/ScamSense.mlflow/#/experiments/0/runs/4e925ae7ab3646a2bfc92ff037381cf2
🧪 View experiment at: https://dagshub.com/Bhoovika/ScamSense.mlflow/#/experiments/0
🏃 View run mbert-scamsense-baseline-ta at: https://dagshub.com/Bhoovika/ScamSense.mlflow/#/experiments/0/runs/e2d1117c21844a1686fe6d2440e28efe
🧪 View experiment at: https://dagshub.com/Bhoovika/ScamSen

## Cell 5 — GitHub Actions CI/CD Workflow

This cell writes a `.github/workflows/ci.yml` file to `/kaggle/working/`. After downloading it, commit it to your ScamSense repo at `.github/workflows/ci.yml`.

The workflow triggers on every push to `main` and:
1. Lints Python files with `flake8`
2. Validates that `model_comparison.csv` exists and has the expected columns
3. Runs a smoke-test of the drift detection logic
4. Posts a status badge to your README

In [14]:
GH_ACTIONS_YML = '''
name: ScamSense CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  lint-and-validate:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repo
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install dependencies
        run: |
          pip install flake8 pandas evidently

      - name: Lint with flake8
        run: |
          # Stop build on syntax errors or undefined names
          flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics
          # Warn on style issues (non-blocking)
          flake8 . --count --exit-zero --max-complexity=10 --max-line-length=120 --statistics

      - name: Validate model_comparison.csv
        run: |
          python - <<EOF
          import pandas as pd, sys
          df = pd.read_csv("results/model_comparison.csv")
          required = {"model", "language", "accuracy", "f1", "precision", "recall", "auc"}
          missing  = required - set(df.columns)
          if missing:
              print(f"FAIL: missing columns: {missing}")
              sys.exit(1)
          if df["model"].nunique() < 2:
              print("FAIL: expected at least 2 models in model_comparison.csv")
              sys.exit(1)
          print(f"PASS: {len(df)} rows, {df['model'].nunique()} models")
          EOF

      - name: Smoke-test drift detection
        run: |
          python - <<EOF
          import pandas as pd
          from evidently.report import Report
          from evidently.metric_preset import DataDriftPreset

          df = pd.read_csv("results/model_comparison.csv")
          METRIC_COLS = ["accuracy", "f1", "precision", "recall", "auc"]
          ref = df[df["model"] == "mbert-scamsense-baseline"][METRIC_COLS]
          cur = df[df["model"] == "xlmroberta-scamsense"][METRIC_COLS]

          report = Report(metrics=[DataDriftPreset()])
          report.run(reference_data=ref, current_data=cur)
          print("PASS: Evidently drift report ran successfully")
          EOF
'''

workflow_path = OUTPUT_DIR / 'github_actions_workflow.yml'
workflow_path.write_text(GH_ACTIONS_YML.strip())
print(f'GitHub Actions workflow saved -> {workflow_path}')
print()
print('Next steps:')
print('  1. Download this file from the Kaggle Output tab')
print('  2. In your ScamSense repo, create the folder: .github/workflows/')
print('  3. Save the file as: .github/workflows/ci.yml')
print('  4. Also move model_comparison.csv to: results/model_comparison.csv')
print('  5. git add . && git commit -m "Add CI/CD workflow" && git push')
print('  6. Check runs at: https://github.com/Bhoovika/ScamSense/actions')


GitHub Actions workflow saved -> /kaggle/working/github_actions_workflow.yml

Next steps:
  1. Download this file from the Kaggle Output tab
  2. In your ScamSense repo, create the folder: .github/workflows/
  3. Save the file as: .github/workflows/ci.yml
  4. Also move model_comparison.csv to: results/model_comparison.csv
  5. git add . && git commit -m "Add CI/CD workflow" && git push
  6. Check runs at: https://github.com/Bhoovika/ScamSense/actions


## Cell 6 — GCP Cloud Run: Architecture Notes + Dockerfile

> **Not executed** — GCP account not set up yet. This cell documents the deployment architecture and writes the `Dockerfile` for when NB09 (Telegram bot) is complete.

### Architecture

```
Telegram User
     │  HTTPS
     ▼
Telegram Bot API
     │  webhook
     ▼
GCP Cloud Run (scamsense-bot)
     │  loads model
     ├─► HuggingFace Hub: Bhoovika/scamsense-xlmroberta
     │  writes prediction
     └─► Supabase: predictions table
```

### Deployment steps (for NB12)
1. `gcloud run deploy scamsense-bot --source . --region asia-southeast1`
2. Set Telegram webhook: `https://api.telegram.org/bot<TOKEN>/setWebhook?url=<CLOUD_RUN_URL>`
3. Add secrets via `gcloud secrets create`

In [15]:
DOCKERFILE = '''
# ScamSense Telegram Bot — Cloud Run Deployment
# Build: docker build -t scamsense-bot .
# Run locally: docker run -p 8080:8080 --env-file .env scamsense-bot

FROM python:3.12-slim

WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    gcc \\
    && rm -rf /var/lib/apt/lists/*

# Install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Cloud Run expects the app to listen on PORT (default 8080)
ENV PORT=8080

# Start the FastAPI webhook server
CMD ["uvicorn", "bot.webhook:app", "--host", "0.0.0.0", "--port", "8080"]
'''

REQUIREMENTS = '''
# ScamSense Bot requirements
fastapi>=0.110.0
uvicorn>=0.29.0
transformers>=4.40.0
torch>=2.2.0
huggingface-hub>=0.22.0
python-telegram-bot>=21.0
supabase>=2.0.0
'''

CLOUDRUN_DEPLOY_SH = '''
#!/bin/bash
# deploy.sh — run from repo root after installing gcloud CLI
# Usage: bash deploy.sh

PROJECT_ID="your-gcp-project-id"      # replace
REGION="asia-southeast1"               # Singapore
SERVICE="scamsense-bot"

gcloud config set project $PROJECT_ID

# Build and deploy
gcloud run deploy $SERVICE \\
  --source . \\
  --region $REGION \\
  --platform managed \\
  --allow-unauthenticated \\
  --set-secrets="TELEGRAM_TOKEN=telegram-token:latest,SUPABASE_URL=supabase-url:latest,SUPABASE_KEY=supabase-key:latest"

echo "Deployed! Set Telegram webhook:"
echo "https://api.telegram.org/bot<TELEGRAM_TOKEN>/setWebhook?url=$(gcloud run services describe $SERVICE --region $REGION --format='value(status.url)')"
'''

# Save all files
(OUTPUT_DIR / 'Dockerfile').write_text(DOCKERFILE.strip())
(OUTPUT_DIR / 'requirements_bot.txt').write_text(REQUIREMENTS.strip())
(OUTPUT_DIR / 'deploy.sh').write_text(CLOUDRUN_DEPLOY_SH.strip())

print('Deployment files saved to /kaggle/working/:')
print('  Dockerfile             <- Cloud Run container definition')
print('  requirements_bot.txt   <- Bot Python dependencies')
print('  deploy.sh              <- One-command GCP deployment script')
print()
print('These will be used in NB12 once the Telegram bot (NB09) is complete.')
print('GCP region: asia-southeast1 (Singapore) — closest to your users')


Deployment files saved to /kaggle/working/:
  Dockerfile             <- Cloud Run container definition
  requirements_bot.txt   <- Bot Python dependencies
  deploy.sh              <- One-command GCP deployment script

These will be used in NB12 once the Telegram bot (NB09) is complete.
GCP region: asia-southeast1 (Singapore) — closest to your users


## Cell 7 — Save outputs + summary

In [16]:
import json

summary = {
    'notebook': 'NB11 Drift Detection + MLflow + CI/CD',
    'outputs': {
        'drift_report':      str(OUTPUT_DIR / 'drift_report.html'),
        'github_actions':    str(OUTPUT_DIR / 'github_actions_workflow.yml'),
        'dockerfile':        str(OUTPUT_DIR / 'Dockerfile'),
        'requirements_bot':  str(OUTPUT_DIR / 'requirements_bot.txt'),
        'deploy_script':     str(OUTPUT_DIR / 'deploy.sh'),
    },
    'mlflow': 'https://dagshub.com/Bhoovika/ScamSense.mlflow',
    'gcp_status': 'documented_not_executed',
    'next': 'NB09 Telegram bot, then NB12 GCP Cloud Run deployment'
}

(OUTPUT_DIR / 'nb11_summary.json').write_text(json.dumps(summary, indent=2))

print('='*60)
print('✅ Notebook 11 complete!')
print()
print('What was done:')
print('  ✅ Evidently drift report   -> drift_report.html')
print('  ✅ MLflow metrics re-logged -> DagsHub (per language + overall)')
print('  ✅ GitHub Actions CI/CD     -> github_actions_workflow.yml')
print('  ✅ GCP Cloud Run docs       -> Dockerfile + deploy.sh')
print()
print('Download from Kaggle Output tab:')
print('  drift_report.html              (open in browser)')
print('  github_actions_workflow.yml    (commit to .github/workflows/ci.yml)')
print('  Dockerfile + deploy.sh         (use in NB12 after NB09 bot is done)')
print()
print('MLflow runs: https://dagshub.com/Bhoovika/ScamSense.mlflow')
print('GitHub Actions (after commit): https://github.com/Bhoovika/ScamSense/actions')
print('='*60)


✅ Notebook 11 complete!

What was done:
  ✅ Evidently drift report   -> drift_report.html
  ✅ MLflow metrics re-logged -> DagsHub (per language + overall)
  ✅ GitHub Actions CI/CD     -> github_actions_workflow.yml
  ✅ GCP Cloud Run docs       -> Dockerfile + deploy.sh

Download from Kaggle Output tab:
  drift_report.html              (open in browser)
  github_actions_workflow.yml    (commit to .github/workflows/ci.yml)
  Dockerfile + deploy.sh         (use in NB12 after NB09 bot is done)

MLflow runs: https://dagshub.com/Bhoovika/ScamSense.mlflow
GitHub Actions (after commit): https://github.com/Bhoovika/ScamSense/actions
